# Modellbewertung mit Iris Dataset - Versicolor vs. Rest 🌸

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klar74/WS2026_lecture/blob/main/Vorlesung_18/model_evaluation_iris_versicolor_vs_rest.ipynb)

## Willkommen zur Modellbewertung!

Heute vergleichen wir zwei Modelle und lernen, ihre Ergebnisse zu verstehen. Wir verwenden das Iris-Dataset und machen daraus ein binäres Klassifikationsproblem: **"Ist es eine Versicolor oder nicht?"**

### Was wir heute lernen:
1. 🔄 **Training und Vergleich trennen** - Lernen und Ausprobieren mit verschiedenen Daten
2. 🔀 **Cross-Validation** - Ergebnisse bei fünf Aufteilungen vergleichen
3. 📊 **Konfusionsmatrix** - Wo macht unser Modell Fehler?
4. 📈 **Metriken verstehen** - Accuracy, Precision, Recall, F1-Score
5. 📉 **ROC-Kurven** - Schwellenwert-Optimierung
6. ⚖️ **Anwendungskontext** - Wann welche Metrik wichtig ist

**Los geht's!** 🚀

In [ ]:
# Schritt 1: Alle benötigten Bibliotheken importieren
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix, classification_report, 
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score, precision_recall_curve
)
import warnings
warnings.filterwarnings('ignore')

# Für schönere Plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("🎉 Alle Bibliotheken erfolgreich geladen!")
print("Wir sind bereit, die Modellvorhersagen zu untersuchen!")

## 🌺 Schritt 1: Daten laden und zu binärem Problem umwandeln

Das Iris-Dataset hat 3 Klassen (Setosa, Versicolor, Virginica). Wir machen daraus ein **binäres Problem**: Versicolor vs. alle anderen.

**Warum Versicolor?** Versicolor ist schwieriger zu unterscheiden als Setosa - perfekt um zu lernen, wie Modelle mit herausfordernderen Problemen umgehen!

In [ ]:
# Iris-Dataset laden
iris = load_iris()
X = iris.data
y_multiclass = iris.target  # Original: 0=Setosa, 1=Versicolor, 2=Virginica

# Zu binärem Problem umwandeln: Versicolor (1) vs. Rest (0)
y_binary = (y_multiclass == 1).astype(int)  # 1 wenn Versicolor, 0 sonst

# Als DataFrame für bessere Übersicht
df = pd.DataFrame(X, columns=iris.feature_names)
df['Original_Class'] = [iris.target_names[i] for i in y_multiclass]
df['Binary_Target'] = y_binary
df['Binary_Label'] = df['Binary_Target'].map({1: 'Versicolor', 0: 'Not_Versicolor'})

print(f"📊 Dataset-Info:")
print(f"   Anzahl Blumen: {len(df)}")
print(f"   Features: {list(iris.feature_names)}")
print(f"\n🎯 Binäre Klassenverteilung:")
print(df['Binary_Label'].value_counts())
print(f"\n📈 Prozentuale Verteilung:")
print(df['Binary_Label'].value_counts(normalize=True) * 100)

# Erste 5 Zeilen anzeigen
print(f"\n🔍 Erste 5 Datensätze:")
df.head()

## 📊 Datenvisualisierung - Können wir die Klassen unterscheiden?

Schauen wir uns an, ob Versicolor gut von den anderen unterscheidbar ist:

In [ ]:
# Visualisierung der Features
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Iris Features: Versicolor vs. Rest', fontsize=16, fontweight='bold')

features = iris.feature_names
colors = ['red', 'blue']
labels = ['Versicolor', 'Not Versicolor']

for i, feature in enumerate(features):
    ax = axes[i//2, i%2]
    bins = np.histogram_bin_edges(df[feature], bins=15)
    
    # Histogramme für beide Klassen
    for class_val, color, label in zip([1, 0], colors, labels):
        data = df[df['Binary_Target'] == class_val][feature]
        ax.hist(data, alpha=0.7, color=color, label=label, bins=bins, edgecolor='black')
    
    ax.set_xlabel(feature)
    ax.set_ylabel('Häufigkeit')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistiken
print("📊 FEATURE-MITTELWERTE:")
for feature in features:
    versicolor_mean = df[df['Binary_Target'] == 1][feature].mean()
    other_mean = df[df['Binary_Target'] == 0][feature].mean()
    print(f"   {feature:20}: Versicolor={versicolor_mean:.2f}, Rest={other_mean:.2f}, Differenz={abs(versicolor_mean-other_mean):.2f}")

print("Die Nicht-Versicolor-Klasse enthält zwei Arten. Manche Histogramme haben deshalb zwei Häufungen.")
print("Abstand der Mittelwerte ist ggf. kein gutes Kriterium für Trennbarkeit.")

## 🔄 Schritt 2: Training und Vergleich aufteilen

Wir teilen die Daten **vor Skalierung und Training** auf. Mit den Trainingsdaten lernen die Modelle. Mit den Vergleichsdaten vergleichen wir danach Modelle und Schwellen. Weil wir sie zum Auswählen nutzen, sind sie keine unabhängige Abschlussprüfung. Stratifizierung bedeutet: Beide Gruppen behalten hier denselben Anteil an Versicolors.

In [ ]:
# Training/Vergleich (80/20) mit Stratifizierung
X_train, X_compare, y_train, y_compare = train_test_split(
    X, y_binary,
    test_size=0.2,          # 20% für den Vergleich
    random_state=42,        # Reproduzierbare Ergebnisse
    stratify=y_binary       # Gleiche Klassenverteilung in Training und Vergleich
)

print("📊 TRAINING/VERGLEICH:")
print(f"   Training: {X_train.shape[0]} Blumen")
print(f"   Vergleich: {X_compare.shape[0]} Blumen")
print(f"   Verhältnis: {X_train.shape[0]/X_compare.shape[0]:.1f}:1 (Training:Vergleich)")

# Klassenverteilung überprüfen
print(f"\n🎯 KLASSENVERTEILUNG:")
print(f"   Training - Versicolor: {np.sum(y_train)}, Not-Versicolor: {len(y_train) - np.sum(y_train)}")
print(f"   Vergleich - Versicolor: {np.sum(y_compare)}, Not-Versicolor: {len(y_compare) - np.sum(y_compare)}")
print(f"   Training % Versicolor: {np.mean(y_train)*100:.1f}%")
print(f"   Vergleich % Versicolor: {np.mean(y_compare)*100:.1f}%")

print("\n✅ Gleiche Klassenanteile in Training und Vergleich!")
print("Die Vergleichsdaten dienen zum Ausprobieren, nicht als unabhängige Abschlussprüfung.")

## 🔧 Schritt 3: Pipeline aufbauen (Skalierung nur aus Trainingsdaten)

**Pipeline = Skalierung + Modell in einem Schritt**

**Warum Pipeline?** Sie lernt die Skalierung nur aus den Daten, mit denen sie gerade trainiert wird. Das gilt auch bei der Cross-Validation. Sie schützt aber nicht automatisch vor allen Fehlern beim Bewerten.

In [ ]:
# Pipeline erstellen: StandardScaler + LogisticRegression
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

print("🔧 PIPELINE AUFGEBAUT:")
print("   1. StandardScaler - Features auf Mittelwert=0, Std=1 skalieren")
print("   2. LogisticRegression - Lineare Klassifikation mit Wahrscheinlichkeiten")
print("\n✅ Beim Training lernt die Pipeline die Skalierung nur aus den Trainingsdaten.")

# Zusätzlich: Random Forest Pipeline für Vergleich
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=100))
])

print("\n🌳 ZUSÄTZLICHE PIPELINE für Vergleich:")
print("   Random Forest - Ensemble von Entscheidungsbäumen")

## 🔀 Schritt 4: Cross-Validation - Fünf Aufteilungen vergleichen

**Eine einzelne Aufteilung kann Glück oder Pech haben.** Wir teilen die Trainingsdaten in fünf Teile. Jeder Teil wird einmal zum Prüfen verwendet, während das Modell mit den anderen vier Teilen lernt. Die Vergleichsdaten bleiben dabei außen vor.

In [ ]:
# 5-Fold Stratified Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Verschiedene Metriken testen
scoring_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

print("🔀 CROSS-VALIDATION ERGEBNISSE (5-Fold):")
print("="*60)

cv_results = {}
for metric in scoring_metrics:
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring=metric)
    cv_results[metric] = scores
    
    print(f"{metric.upper():12}: {scores.mean():.3f} ± {scores.std():.3f} | Alle Folds: {[f'{s:.3f}' for s in scores]}")

print("="*60)
print("📊 INTERPRETATION:")
print(f"   Accuracy:  {cv_results['accuracy'].mean():.1%} der Vorhersagen sind korrekt")
print(f"   Precision: {cv_results['precision'].mean():.1%} der 'Versicolor'-Vorhersagen stimmen im Mittel")
print(f"   Recall:    {cv_results['recall'].mean():.1%} aller Versicolors werden im Mittel erkannt")
print(f"   F1-Score:  {cv_results['f1'].mean():.3f} (harmonisches Mittel von Precision & Recall)")
print(f"   ROC-AUC:   {cv_results['roc_auc'].mean():.3f} (0.5=zufällig, 1.0=perfekt)")

# Schwankungen der fünf Ergebnisse zeigen
print("\n🎯 SCHWANKUNGEN ZWISCHEN DEN FÜNF TEILEN:")
print("Die Standardabweichung zeigt, wie stark die fünf Werte schwanken. Kleiner bedeutet hier weniger Schwankung.")
for metric in scoring_metrics:
    std_dev = cv_results[metric].std()
    print(f"   {metric:12}: Standardabweichung {std_dev:.3f}")

## 🏋️ Schritt 5: Modell auf Trainingsdaten trainieren

Jetzt trainieren wir das finale Modell mit allen Trainingsdaten:

In [ ]:
# Modell trainieren
print("🏋️ Trainiere das finale Modell...")
pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

# Vorhersagen für Vergleichsdaten
y_pred = pipeline.predict(X_compare)
y_pred_proba = pipeline.predict_proba(X_compare)[:, 1]  # Vorhersagewert für Klasse 1 (Versicolor)

y_pred_rf = rf_pipeline.predict(X_compare)
y_pred_proba_rf = rf_pipeline.predict_proba(X_compare)[:, 1]

print("✅ Modelle trainiert!")
print(f"\n🔍 FEATURE-WICHTIGKEIT (Logistic Regression):")
feature_importance = np.abs(pipeline.named_steps['classifier'].coef_[0])
feature_ranks = sorted(zip(iris.feature_names, feature_importance), key=lambda x: x[1], reverse=True)
for i, (feature, importance) in enumerate(feature_ranks):
    print(f"   {i+1}. {feature:25}: {importance:.3f}")

print(f"\n🔍 FEATURE-WICHTIGKEIT (Random Forest):")
feature_importance_rf = rf_pipeline.named_steps['classifier'].feature_importances_
feature_ranks = sorted(zip(iris.feature_names, feature_importance_rf), key=lambda x: x[1], reverse=True)
for i, (feature, importance) in enumerate(feature_ranks):
    print(f"   {i+1}. {feature:25}: {importance:.3f}")

## 📊 Schritt 6: Konfusionsmatrix - Wo macht unser Modell Fehler?

Die Konfusionsmatrix zeigt uns **genau**, welche Fehler unser Modell macht:

In [ ]:
# Konfusionsmatrix berechnen
cm = confusion_matrix(y_compare, y_pred)
cm_rf = confusion_matrix(y_compare, y_pred_rf)

# Schöne Visualisierung
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Logistic Regression
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
           xticklabels=['Not Versicolor', 'Versicolor'], 
           yticklabels=['Not Versicolor', 'Versicolor'], ax=ax1)
ax1.set_title('Konfusionsmatrix - Logistic Regression', fontweight='bold')
ax1.set_xlabel('Vorhersage')
ax1.set_ylabel('Tatsächlich')

# Random Forest
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', 
           xticklabels=['Not Versicolor', 'Versicolor'], 
           yticklabels=['Not Versicolor', 'Versicolor'], ax=ax2)
ax2.set_title('Konfusionsmatrix - Random Forest', fontweight='bold')
ax2.set_xlabel('Vorhersage')
ax2.set_ylabel('Tatsächlich')

plt.tight_layout()
plt.show()

# Matrix interpretieren
tn, fp, fn, tp = cm.ravel()
tn_rf, fp_rf, fn_rf, tp_rf = cm_rf.ravel()

print("📊 KONFUSIONSMATRIX INTERPRETATION (Logistic Regression):")
print(f"   True Positives (TP):  {tp:2d} - Versicolors korrekt als Versicolor erkannt")
print(f"   True Negatives (TN):  {tn:2d} - Nicht-Versicolors korrekt als Nicht-Versicolor erkannt")
print(f"   False Positives (FP): {fp:2d} - Nicht-Versicolors fälschlich als Versicolor erkannt (Fehlalarm)")
print(f"   False Negatives (FN): {fn:2d} - Versicolors übersehen (verpasste Erkennung)")

print(f"\n📊 KONFUSIONSMATRIX INTERPRETATION (Random Forest):")
print(f"   True Positives (TP):  {tp_rf:2d} - Versicolors korrekt als Versicolor erkannt")
print(f"   True Negatives (TN):  {tn_rf:2d} - Nicht-Versicolors korrekt als Nicht-Versicolor erkannt")
print(f"   False Positives (FP): {fp_rf:2d} - Nicht-Versicolors fälschlich als Versicolor erkannt (Fehlalarm)")
print(f"   False Negatives (FN): {fn_rf:2d} - Versicolors übersehen (verpasste Erkennung)")

## 📈 Schritt 7: Alle Metriken berechnen und verstehen

Jetzt berechnen wir alle wichtigen Metriken und verstehen, was sie bedeuten:

In [ ]:
# Alle Metriken für beide Modelle berechnen
def calculate_all_metrics(y_true, y_pred, model_name):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    # Zusätzliche Metriken
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    print(f"\n📊 {model_name} - ALLE METRIKEN:")
    print("="*50)
    print(f"   Accuracy:     {accuracy:.3f} = {accuracy:.1%} aller Vorhersagen korrekt")
    print(f"   Precision:    {precision:.3f} = {precision:.1%} der 'Versicolor'-Vorhersagen stimmen")
    print(f"   Recall:       {recall:.3f} = {recall:.1%} aller Versicolors werden erkannt")
    print(f"   F1-Score:     {f1:.3f} = Harmonisches Mittel von Precision & Recall")
    print(f"   Specificity:  {specificity:.3f} = {specificity:.1%} der Nicht-Versicolors korrekt erkannt")
    print(f"   FPR:          {false_positive_rate:.3f} = {false_positive_rate:.1%} Fehlalarm-Rate")
    
    # Interpretation
    print(f"\n💡 PRAKTISCHE BEDEUTUNG:")
    if precision > 0.95:
        print("   🎯 Auf den Vergleichsdaten stimmen fast alle 'Versicolor'-Meldungen.")
    if recall > 0.95:
        print("   🔍 Auf den Vergleichsdaten werden fast alle Versicolors erkannt.")
    if f1 > 0.95:
        print("   ⚖️ Auf den Vergleichsdaten sind Precision und Recall beide hoch.")
    
    return {
        'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1,
        'specificity': specificity, 'fpr': false_positive_rate
    }

# Metriken für beide Modelle
lr_metrics = calculate_all_metrics(y_compare, y_pred, "LOGISTIC REGRESSION")
rf_metrics = calculate_all_metrics(y_compare, y_pred_rf, "RANDOM FOREST")

baseline_accuracy = np.mean(y_compare == 0)
print(f"\n💡 Immer 'Nicht-Versicolor' vorhersagen würde schon {baseline_accuracy:.1%} Accuracy erreichen.")
print("Dabei würde aber keine einzige Versicolor erkannt. Deshalb schauen wir auch auf Recall und Precision.")

# Detaillierter Classification Report
print(f"\n📋 DETAILLIERTER CLASSIFICATION REPORT:")
print("\nLogistic Regression:")
print(classification_report(y_compare, y_pred, target_names=['Not Versicolor', 'Versicolor']))
print("\nRandom Forest:")
print(classification_report(y_compare, y_pred_rf, target_names=['Not Versicolor', 'Versicolor']))

## 📉 Schritt 8: ROC-Kurve - Schwellenwert-Optimierung verstehen

**Die ROC-Kurve zeigt, wie sich Treffer und Fehlalarme bei verschiedenen Schwellen verändern.**

- **X-Achse:** False Positive Rate (Fehlalarm-Rate)
- **Y-Achse:** True Positive Rate = Recall (Erkennungsrate)

In [ ]:
# ROC-Kurven für beide Modelle
fpr_lr, tpr_lr, thresholds_lr = roc_curve(y_compare, y_pred_proba)
fpr_rf, tpr_rf, thresholds_rf = roc_curve(y_compare, y_pred_proba_rf)

auc_lr = roc_auc_score(y_compare, y_pred_proba)
auc_rf = roc_auc_score(y_compare, y_pred_proba_rf)

# ROC-Kurve plotten
plt.figure(figsize=(12, 5))

# ROC-Kurve
plt.subplot(1, 2, 1)
plt.plot(fpr_lr, tpr_lr, color='blue', lw=2, label=f'Logistic Regression (AUC = {auc_lr:.3f})')
plt.plot(fpr_rf, tpr_rf, color='green', lw=2, label=f'Random Forest (AUC = {auc_rf:.3f})')
plt.plot([0, 1], [0, 1], color='red', lw=2, linestyle='--', label='Zufälliges Raten (AUC = 0.5)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (Fehlalarm-Rate)')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC-Kurve: Schwellenwert-Performance', fontweight='bold')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)

# Precision-Recall Kurve
plt.subplot(1, 2, 2)
precision_lr, recall_lr, _ = precision_recall_curve(y_compare, y_pred_proba)
precision_rf, recall_rf, _ = precision_recall_curve(y_compare, y_pred_proba_rf)

plt.plot(recall_lr, precision_lr, color='blue', lw=2, label=f'Logistic Regression')
plt.plot(recall_rf, precision_rf, color='green', lw=2, label=f'Random Forest')
plt.axhline(y=np.mean(y_compare), color='red', linestyle='--', label=f'Baseline (Zufall): {np.mean(y_compare):.3f}')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall (Erkennungsrate)')
plt.ylabel('Precision (Genauigkeit der Vorhersagen)')
plt.title('Precision-Recall Kurve', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"📊 ROC-AUC INTERPRETATION:")
print(f"   Logistic Regression AUC: {auc_lr:.3f}")
print(f"   Random Forest AUC:       {auc_rf:.3f}")
print("\n💡 Eine höhere AUC bedeutet hier: Versicolors werden besser vor Nicht-Versicolors eingeordnet.")
print("Welche Fehler im Einsatz akzeptabel sind, sagt die AUC allein nicht.")

# Modell mit höchster AUC in diesem Vergleich bestimmen
best_model = "Logistic Regression" if auc_lr > auc_rf else "Random Forest"
best_auc = max(auc_lr, auc_rf)
if auc_lr == auc_rf:
    print("\n🏆 Beide Modelle haben dieselbe AUC; für die Schlussübersicht verwenden wir den Random Forest.")
else:
    print(f"\n🏆 HÖCHSTE AUC IM VERGLEICH: {best_model} mit AUC = {best_auc:.3f}")

## ⚖️ Schritt 9: Schwellenwert-Optimierung für verschiedene Anwendungsfälle

**Standard-Schwellenwert ist 0.5** - für andere Ziele können andere Schwellen sinnvoll sein.

Wir probieren Schwellen für die logistische Regression auf den Vergleichsdaten aus. Bei gleichem Zielwert bevorzugen wir weniger Fehlalarme, danach weniger übersehene Versicolors. Die Ergebnisse gelten für diese Übung, nicht als unabhängige Abschlussprüfung.

In [ ]:
def find_best_tested_threshold(y_true, y_proba, objective='f1'):
    """Wählt nach Zielwert, dann nach weniger Fehlalarmen und weniger übersehenen Fällen."""
    if objective not in ('f1', 'precision', 'recall'):
        raise ValueError('Als Ziel sind f1, precision und recall erlaubt.')
    thresholds = np.arange(0.1, 1.0, 0.01)
    best_score = -np.inf
    best_threshold = None
    best_counts = None
    best_key = (-np.inf, -np.inf, -np.inf)
    
    scores = []
    for threshold in thresholds:
        y_pred_thresh = (y_proba >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred_thresh, labels=[0, 1]).ravel()
        
        if objective == 'f1':
            score = f1_score(y_true, y_pred_thresh, zero_division=0)
        elif objective == 'precision':
            score = precision_score(y_true, y_pred_thresh) if tp + fp > 0 else np.nan
        elif objective == 'recall':
            score = recall_score(y_true, y_pred_thresh)
        
        scores.append(score)
        if np.isnan(score):
            continue
        candidate_key = (score, -fp, -fn)
        if candidate_key > best_key:
            best_key = candidate_key
            best_score = score
            best_threshold = threshold
            best_counts = {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}
    
    if best_counts is None:
        raise ValueError('Für dieses Ziel liefert keine der ausprobierten Schwellen einen definierten Wert.')
    return best_threshold, best_score, thresholds, scores, best_counts

# Verschiedene Optimierungsziele
objectives = ['f1', 'precision', 'recall']
colors = ['blue', 'red', 'green']

plt.figure(figsize=(15, 5))

results = {}
for i, (obj, color) in enumerate(zip(objectives, colors)):
    threshold, score, thresholds, scores, counts = find_best_tested_threshold(y_compare, y_pred_proba, obj)
    results[obj] = {'threshold': threshold, 'score': score, **counts}
    
    plt.subplot(1, 3, i+1)
    plt.plot(thresholds, scores, color=color, linewidth=2)
    plt.axvline(threshold, color='red', linestyle='--', alpha=0.7)
    plt.axvline(0.5, color='gray', linestyle=':', alpha=0.7, label='Standard (0.5)')
    plt.xlabel('Schwellenwert')
    plt.ylabel(f'{obj.upper()}-Score')
    plt.title(f'Ziel: {obj.upper()}\nBeste getestete Schwelle: {threshold:.2f}\nScore: {score:.3f}', fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.legend()

plt.tight_layout()
plt.show()

print("⚖️ SCHWELLEN VERGLEICHEN (LOGISTISCHE REGRESSION):")
print("="*60)
for obj in objectives:
    result = results[obj]
    print(f"{obj.upper():10}: Beste ausprobierte Schwelle = {result['threshold']:.3f}, Score = {result['score']:.3f}")
    print(f"   Erkannt: {result['tp']}, übersehen: {result['fn']}, Fehlalarme: {result['fp']}, korrekt als Rest erkannt: {result['tn']}")
print("Bei gleichem Zielwert wählen wir weniger Fehlalarme, danach weniger übersehene Fälle.")
print("Ohne Versicolor-Meldungen ist Precision nicht definiert; diese Stellen bleiben im Diagramm leer.")

print(f"\nStandard:  Schwellenwert = 0.500")

# Praktische Szenarien
print(f"\n🎯 PRAKTISCHE ANWENDUNG:")
print(f"\n1. 🔬 BOTANISCHE FORSCHUNG: 'Keine Versicolor übersehen!'")
print(f"   → Ziel RECALL: Schwellenwert = {results['recall']['threshold']:.3f}")
print(f"   → Erkenne {results['recall']['score']:.1%} aller Versicolors")
print(f"   → Dabei entstehen {results['recall']['fp']} Fehlalarme auf den Vergleichsdaten.")

print("\n2. 🏷️ AUTOMATISCHE ETIKETTIERUNG: 'Möglichst wenige falsche Versicolor-Labels!'")
print(f"   → Ziel PRECISION: Schwellenwert = {results['precision']['threshold']:.3f}")
print(f"   → {results['precision']['score']:.1%} der 'Versicolor'-Labels sind korrekt")
print(f"   → Dabei werden {results['precision']['fn']} Versicolors übersehen.")

print("\n3. ⚖️ F1-SCORE: Precision und Recall gemeinsam betrachten")
print(f"   → Ziel F1: Schwellenwert = {results['f1']['threshold']:.3f}")
print(f"   → F1-Score = {results['f1']['score']:.3f}")
print(f"   → Fehlalarme: {results['f1']['fp']}, übersehene Versicolors: {results['f1']['fn']}")

## 🔬 Schritt 10: Fehleranalyse - Welche Blumen werden falsch klassifiziert?

Wir untersuchen die logistische Regression mit der Standardschwelle 0.5. Ein Vorhersagewert nahe 0.5 liegt nahe an der Entscheidungsschwelle. Er ist aber keine Garantie dafür, ob die Vorhersage richtig ist.

In [ ]:
# Vergleichsdaten und Vorhersagen bei Standardschwelle 0.5 analysieren
compare_results = pd.DataFrame(X_compare, columns=iris.feature_names)
compare_results['True_Class'] = y_compare
compare_results['Predicted_Class'] = y_pred
compare_results['Predicted_Proba'] = y_pred_proba
compare_results['Correct'] = (y_compare == y_pred)
compare_results['True_Label'] = compare_results['True_Class'].map({1: 'Versicolor', 0: 'Not_Versicolor'})
compare_results['Pred_Label'] = compare_results['Predicted_Class'].map({1: 'Versicolor', 0: 'Not_Versicolor'})

# Falsch klassifizierte Beispiele
false_predictions = compare_results[~compare_results['Correct']]

print("🔬 FEHLERANALYSE DER LOGISTISCHEN REGRESSION (SCHWELLE 0.5):")
print(f"   Vergleichsblumen: {len(compare_results)}")
print(f"   Korrekt: {compare_results['Correct'].sum()}")
print(f"   Falsch: {len(false_predictions)}")
print(f"   Accuracy: {compare_results['Correct'].mean():.1%}")

if len(false_predictions) > 0:
    print(f"\n❌ FALSCH KLASSIFIZIERTE BLUMEN:")
    print(false_predictions[['True_Label', 'Pred_Label', 'Predicted_Proba', 
                            'sepal length (cm)', 'sepal width (cm)', 
                            'petal length (cm)', 'petal width (cm)']].to_string(index=False))
    
    # Analyse der Fehlklassifikationen
    false_positives = false_predictions[false_predictions['True_Class'] == 0]
    false_negatives = false_predictions[false_predictions['True_Class'] == 1]
    
    print(f"\n📊 FEHLERTYPEN:")
    print(f"   False Positives: {len(false_positives)} (Nicht-Versicolor als Versicolor klassifiziert)")
    print(f"   False Negatives: {len(false_negatives)} (Versicolor als Nicht-Versicolor klassifiziert)")
    
    if len(false_positives) > 0:
        print(f"\n🚨 FALSE POSITIVE ANALYSE:")
        print(f"   Durchschnittlicher Vorhersagewert für Versicolor: {false_positives['Predicted_Proba'].mean():.3f}")
        print(f"   Abstand zur Schwelle 0.5 im Mittel: {(false_positives['Predicted_Proba'] - 0.5).abs().mean():.3f}")
    
    if len(false_negatives) > 0:
        print(f"\n🚨 FALSE NEGATIVE ANALYSE:")
        print(f"   Durchschnittlicher Vorhersagewert für Versicolor: {false_negatives['Predicted_Proba'].mean():.3f}")
        print(f"   Abstand zur Schwelle 0.5 im Mittel: {(false_negatives['Predicted_Proba'] - 0.5).abs().mean():.3f}")
else:
    print(f"\n🎉 PERFEKTE KLASSIFIKATION!")
    print("   Alle Vergleichsblumen wurden korrekt klassifiziert!")

# Vorhersagewerte nach ihrem Abstand zur Entscheidungsschwelle gruppieren
print("\n🎯 ABSTAND ZUR SCHWELLE 0.5:")
high_confidence = compare_results[(compare_results['Predicted_Proba'] > 0.9) | (compare_results['Predicted_Proba'] < 0.1)]
medium_confidence = compare_results[(compare_results['Predicted_Proba'] >= 0.7) & (compare_results['Predicted_Proba'] <= 0.9) |
                                   (compare_results['Predicted_Proba'] >= 0.1) & (compare_results['Predicted_Proba'] <= 0.3)]
low_confidence = compare_results[(compare_results['Predicted_Proba'] > 0.3) & (compare_results['Predicted_Proba'] < 0.7)]

for label, group in [('Weit entfernt (<0.1 oder >0.9)', high_confidence),
                     ('Dazwischen (0.1-0.3 oder 0.7-0.9)', medium_confidence),
                     ('Nahe an 0.5 (0.3-0.7)', low_confidence)]:
    if group.empty:
        print(f"   {label}: keine Beispiele in dieser Gruppe")
    else:
        print(f"   {label}: {len(group)} Blumen, Trefferquote: {group['Correct'].mean():.1%}")

## 🎉 Zusammenfassung und wichtige Erkenntnisse

**Was wir heute gelernt haben:**

In [ ]:
# Finale Zusammenfassung
print("🎉 MODELLBEWERTUNG KOMPLETT!")
print("="*60)
print(f"\n📊 FINALE ERGEBNISSE:")
print(f"   Dataset: Iris (Versicolor vs. Rest)")
print(f"   Trainingssamples: {len(X_train)}")
print(f"   Vergleichsblumen: {len(X_compare)}")
print(f"   Klassenverteilung: {np.mean(y_compare):.1%} Versicolor")

selected_metrics = lr_metrics if best_model == 'Logistic Regression' else rf_metrics
print("\n🏆 MODELL MIT HÖCHSTER AUC IN DIESEM VERGLEICH:")
print(f"   Modell: {best_model}")
print(f"   Accuracy: {selected_metrics['accuracy']:.1%}")
print(f"   Precision: {selected_metrics['precision']:.1%}")
print(f"   Recall: {selected_metrics['recall']:.1%}")
print(f"   F1-Score: {selected_metrics['f1']:.3f}")
print(f"   ROC-AUC: {best_auc:.3f}")
print("   Alle Kennzahlen gehören zum genannten Modell; die Daten dienten auch zum Auswählen.")

print(f"\n✅ GELERNTE KONZEPTE:")
print("   ✓ Training und Vergleich mit gleichen Klassenanteilen aufteilen")
print("   ✓ Ergebnisse bei fünf Aufteilungen der Trainingsdaten vergleichen")
print("   ✓ Skalierung nur aus Trainingsdaten lernen")
print(f"   ✓ Konfusionsmatrix interpretieren")
print(f"   ✓ Accuracy vs. Precision vs. Recall verstehen")
print(f"   ✓ ROC-Kurven und AUC bewerten")
print(f"   ✓ Schwellenwerte für verschiedene Anwendungsfälle optimieren")
print("   ✓ Modellvergleich und unabhängige Abschlussprüfung unterscheiden")

print(f"\n🎯 WICHTIGSTE ERKENNTNISSE:")
print(f"   1. Versicolor ist schwieriger zu unterscheiden als Setosa - realistischeres Problem!")
print(f"   2. Die logistische Regression erkennt hier {lr_metrics['recall']:.1%} der Versicolors, der Random Forest {rf_metrics['recall']:.1%}.")
print(f"   3. Schwellenwert-Optimierung kann die Performance für spezielle Anwendungen verbessern")
print("   4. Cross-Validation zeigt, wie die Ergebnisse zwischen den Aufteilungen schwanken")
print("   5. Die Pipeline lernt die Skalierung nur aus Trainingsdaten, verhindert aber nicht jeden Bewertungsfehler")

print(f"\n💡 FÜR DIE PRAXIS:")
print(f"   • Immer mehrere Metriken anschauen, nicht nur Accuracy")
print(f"   • Cross-Validation für robuste Schätzungen nutzen")
print(f"   • Schwellenwerte je nach Anwendungskontext optimieren")
print(f"   • Konfusionsmatrix zeigt, wo das Modell Schwächen hat")
print("   • Nach dem Auswählen braucht eine unabhängige Abschlussprüfung weitere, bisher unbenutzte Daten")
print("   • Bei ungleichen Klassenanteilen: Auch die Precision-Recall-Kurve betrachten")

print("\n" + "="*60)
print("🎓 HERZLICHEN GLÜCKWUNSCH!")
print("Ihr habt wichtige Werkzeuge kennengelernt, um Modellvorhersagen zu vergleichen!")
print("="*60)